In [1]:
from minio import Minio
from minio.error import S3Error
import math
import requests
from requests.auth import HTTPBasicAuth
import json
import zipfile
import time
import os

import jwt
import datetime
import pytz
from datetime import datetime
# Definition of necessary functions

def get_token(text):
    browser=['token":"','","file_stage_in']
    pos=[]
    for n in browser:
        k =text.find(n)
        if k!=-1:
            pos.append(k)
        else:
            break
    if k==-1:
        print('Error in connection')
        return None
    else:
        return text[pos[0]+8:pos[1]]

def get_cpuService(text):
    browser=['cpu":"','","total_memory']
    pos=[]
    for n in browser:
        k=text.find(n)
        if k!=-1:
            pos.append(k)
        else:
            break
    if k==-1:
        print('Error in connection')
        return None
    else:
        return 1000*float(text[pos[0]+6:pos[1]])

def get_memoryService(text):
    browser=['memory":"','Gi']
    pos=[]
    for n in browser:
        k=text.find(n)
        if k!=-1:
            pos.append(k)
        else:
            break
    if k==-1:
        print('Error in connection')
        return None
    else:
        return (float(text[pos[0]+9:pos[1]]))

def connect_to_minio(config):
    MinIO_url = config['url']
    MinIO_access_key = config['access_key']
    MinIO_secret_key = config['secret_key']
    #print(f"Connecting to MinIO at {url_minio} with access key {access_key}")
    return MinIO_url,MinIO_access_key,MinIO_secret_key 

def use_bucket(config):
    bucket_name = config['name']
    folder_prefix = config['folder_prefix']
    #print(f"Using bucket {bucket_name} with folder prefix {folder_prefix}")
    return bucket_name, folder_prefix

def setup_output(config):
    output_file = config['file']
    return output_file

def use_service(config):
    service_name = config['name']
    return service_name

def connect_to_oscar_cluster(config):
    token_cluster=''
    oscar_cluster= config['url']
    if 'username' in config_data.get('oscar_cluster', {}).get('auth_basic', {}):
        username = config['auth_basic']['username']
    if 'password' in config_data.get('oscar_cluster', {}).get('auth_basic', {}):
        password = config['auth_basic']['password']
    if username !="" and password != "":
        basic= True 
    else:
        if 'token' in config_data.get('oscar_cluster', {}).get('auth_token', {}):
            token_cluster = config['auth_token']['refresh_token']
            if token_cluster !='':
                basic=False
             
    return oscar_cluster,username,password,token_cluster,basic
def use_directory(config):
    directory=config['local'].get('folder')
    return directory

def list_directory(directory_path, output_file):
    try:
        # Get the list of files in the directory
        files = os.listdir(directory_path)
        
        # Filter the images with .jpg extension
        images = [file for file in files if file.lower().endswith('.jpg')]
        
        # Display the number of images
        print(f"Found {len(images)} images in the directory '{directory_path}'.")
        
        # Save the names of the images to a text file
        with open(output_file, 'w') as file:
            file.write('\n'.join(images))
        
        print(f"The names of the images have been saved to '{output_file}'.")
    except Exception as e:
        print(f"Error: {e}")
    return len(images), images
def connect_to_oscar_cluster(config):
    refresh_token=''
    oscar_cluster= config['url']
    if 'username' in config_data.get('oscar_cluster', {}).get('auth_basic', {}):
        username = config['auth_basic']['username']
    if 'password' in config_data.get('oscar_cluster', {}).get('auth_basic', {}):
        password = config['auth_basic']['password']
    if username !="" and password != "":
        basic= True 
    else:
        if 'refresh_token' in config_data.get('oscar_cluster', {}).get('auth_token', {}):
            refresh_token = config['auth_token']['refresh_token']
            if refresh_token !='':
                basic=False
             
    return oscar_cluster,username,password,refresh_token,basic
def new_token(url,refresh_token):
    
    data = {
    'grant_type': 'refresh_token',
    'refresh_token':refresh_token,
    'client_id': 'token-portal',
    'scope': 'openid email profile voperson_id eduperson_entitlement'
}


    response = requests.post(url, data=data)


    if response.status_code == 200:
    
        err=response.status_code
        response_data = response.json()
   
    
        access_token = response_data.get('access_token')
        expires_in = response_data.get('expires_in')

    else:
        print(f"Error: {response.status_code}, {response.text}")
        err=response.status_code

    return access_token, err
def expire_token(token):
    try:
   # Decode the token (without verifying the signature)
        decoded_token = jwt.decode(token, options={"verify_signature": False}, algorithms=["HS256"])
        

        # Extract the 'exp' expiration field
        exp_timestamp = decoded_token.get('exp')

        if exp_timestamp:
        
            #exp_datetime = datetime.datetime.utcfromtimestamp(exp_timestamp)
            exp_datetime = datetime.utcfromtimestamp(exp_timestamp)
            timezone_spain = pytz.timezone('Europe/Madrid')

        # Convert UTC time to Spain time zone
            exp_datetime_spain = exp_datetime.replace(tzinfo=pytz.utc).astimezone(timezone_spain)

            #current_time = datetime.datetime.utcnow()
            current_time = datetime.utcnow()
            current = current_time.replace(tzinfo=pytz.utc).astimezone(timezone_spain)
            k=exp_datetime_spain - current

            if k.total_seconds()/60 <0:
                print("Token has expired")
           
        else:
            print("The token has no expiration date")

    except jwt.ExpiredSignatureError:
        print("The token has already expired.")
    except jwt.InvalidTokenError:
        print("Invalid token.")
    return exp_datetime_spain, k    


In [27]:
with open('config-walton-direct.json', 'r') as config_file:
    config_data = json.load(config_file)


# Take configuration values
MinIO_url,MinIO_access_key,MinIO_secret_key = connect_to_minio(config_data['MinIO'])
bucket_name, folder_prefix = use_bucket(config_data['bucket'])
output_file=setup_output(config_data['output'])
service_name=use_service(config_data['service'])
oscar_cluster, username, password,refresh_token, basic = connect_to_oscar_cluster(config_data['oscar_cluster'])



In [30]:
##invoke an isolated zip file

url = 'https://aai.egi.eu/auth/realms/egi/protocol/openid-connect/token'
token_cluster,err=new_token(url,refresh_token)

if basic:
    headers = {    
    'Authorization': "Bearer " + token_service,
    'Content-Type': 'application/json',
}
else:
    headers = {
    'Authorization': "Bearer " + token_cluster,
    'Content-Type': 'application/json',
    }

# Ensure the URL is properly constructed (if `oscar_cluster` does not have "https://")
if not oscar_cluster.startswith("https://"):
    url_invoke = "https://" + oscar_cluster + "/job/" + service_name
else:
    url_invoke = oscar_cluster + "/job/" + service_name
    
# zip name to invoke    
name_zip="test.zip"    


print(url_invoke)

data = {
        "zip": name_zip
        
    }
print(data)

try:
    response = requests.post(url_invoke, headers=headers, json=data,verify=True)
    print(response.text)
    print(response.status_code)
    if response.status_code == 200 or response.status_code == 201 :
        print("Services OK")
    else:
        print(response.text)
except Exception as ex:
        print("Error running service: ", ex)
        print(response.text)

https://inference-walton.cloud.imagine-ai.eu/job/fish-detector-zip-3
{'zip': 'prueba.zip'}

201
Services OK
